Notebook này dùng file all_documents_optimized_chunks.jsonl từ file pipeline preprocessing/chunking để thực hiện các việc sau:
1. Đọc và kiểm tra nhanh JSONL.
2. Giữ nguyên text và metadata gốc.
3. Tạo embedding bằng OpenAI.
4. Lưu vào Chroma persistent database.

# **0. Cài đặt thư viện**

In [ ]:
# openai: gọi OpenAI Embedding API.
# chromadb: lưu vector database local.
# pandas: thống kê dữ liệu.
# tqdm: hiển thị thanh tiến trình.
!pip install -q openai chromadb pandas tqdm


# **1. Import thư viện**

=> Khai báo thư viện dùng cho toàn notebook

In [ ]:
# os: đọc biến môi trường OPENAI_API_KEY.
# json: đọc/ghi file JSONL.
# time: nghỉ nhẹ giữa batch để hạn chế rate limit.
# Path: xử lý đường dẫn file/thư mục.
# typing: ghi chú kiểu dữ liệu để code dễ hiểu hơn.
import os
import json
import time
from pathlib import Path
from typing import Any, Dict, List

# pandas dùng để thống kê nhanh file chunks.
import pandas as pd

# tqdm hiển thị progress bar khi embed nhiều chunk.
from tqdm.auto import tqdm

# chromadb là vector database local.
import chromadb

# OpenAI client để gọi Embeddings API.
from openai import OpenAI

# **2. Khai báo cấu hình của notebook**

=> Khai báo đường dẫn, tên DB và tên model

In [ ]:
# File chunk đã hoàn thiện từ pipeline preprocessing/chunking.
JSONL_PATH = "all_documents_optimized_chunks.jsonl"

# Thư mục lưu Chroma DB local. Tên có _openai để tránh nhầm với DB khác.
CHROMA_DIR = "./chroma_ou_rag_db_openai"

# Tên collection trong Chroma.
COLLECTION_NAME = "ou_academic_rag_openai"

# Model embedding OpenAI dùng cho documents và query.
EMBEDDING_MODEL = "text-embedding-3-small"

# Số text mỗi lần gọi embedding API. 16 là mức an toàn cho Jupyter/local.
EMBED_BATCH_SIZE = 16

# File cache embedding để nếu notebook dừng giữa chừng thì chạy lại không embed từ đầu.
EMBEDDING_CACHE_PATH = "embedding_cache_openai.jsonl"

print("JSONL_PATH:", JSONL_PATH)
print("CHROMA_DIR:", CHROMA_DIR)
print("COLLECTION_NAME:", COLLECTION_NAME)
print("EMBEDDING_MODEL:", EMBEDDING_MODEL)


# **3. Thiết lập Open API key**

=> Kiểm tra Open_API_Key và khởi tạo OpenAI client

In [ ]:
import os
from dotenv import load_dotenv

# Đọc biến môi trường từ file .env trong cùng thư mục project
load_dotenv()

if os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY đã sẵn sàng.")
else:
    print("Bạn chưa set OPENAI_API_KEY.")

# Tạo client dùng cho toàn notebook.
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Đã khởi tạo OpenAI client thành công.")

# **4. Đọc file JSONL**

=> Đọc từng dòng JSONL trong file .jsonl

Nếu đúng thì gồm các field quan trọng: id, source, document_name, document_type, chunk_type, page_start, page_end, title, section, article, text, metadata.

In [ ]:
def load_jsonl(path: str) -> List[Dict[str, Any]]:
    # records sẽ chứa toàn bộ chunks sau khi đọc file.
    records = []
    path_obj = Path(path)

    # Báo lỗi rõ nếu file không nằm cùng thư mục notebook.
    if not path_obj.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}. Hãy kiểm tra JSONL_PATH.")

    # Đọc từng dòng vì JSONL = mỗi dòng là một object JSON.
    with path_obj.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Lỗi JSON ở dòng {line_no}: {e}")

    return records

chunks = load_jsonl(JSONL_PATH)
print(f"Số chunks đọc được: {len(chunks)}")
print("Các key của chunk đầu tiên:")
print(list(chunks[0].keys()))


# **5. Thống kê nhanh data**

=> Kiểm tra số lượng theo document_type và chunk_type..

In [ ]:
df = pd.DataFrame(chunks)

print("Thống kê document_type:")
print(df["document_type"].value_counts(dropna=False))

print("Thống kê chunk_type:")
print(df["chunk_type"].value_counts(dropna=False).head(30))

print("Xem nhanh metadata chính:")
display(df[["id", "document_name", "document_type", "chunk_type", "page_start", "page_end"]].head())


# **6. Chuẩn hóa document và metadata cho Chroma**

Dùng để chuẩn bị ids, documents, metadata để đưa vào Chroma. Metadata được giữ cho citation/source

In [ ]:
def clean_metadata_value(value: Any) -> Any:
    # Chroma metadata chỉ nhận str/int/float/bool/None.
    # Nếu gặp list/dict, chuyển sang chuỗi JSON để không mất thông tin.
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    return json.dumps(value, ensure_ascii=False)


def build_metadata(item: Dict[str, Any]) -> Dict[str, Any]:
    # Các field quan trọng dùng để lọc retrieval và hiển thị nguồn.
    important_keys = [
        "source", "document_name", "document_type", "chunk_type", "chunk_id",
        "page_start", "page_end", "title", "section", "article",
        "chapter", "chapter_number", "part_index", "num_parts"
    ]

    meta = {}

    # Lấy metadata chính ở cấp ngoài của chunk.
    for key in important_keys:
        if key in item:
            meta[key] = clean_metadata_value(item.get(key))

    # Giữ thêm metadata gốc nếu có.
    raw_meta = item.get("metadata", {})
    if isinstance(raw_meta, dict):
        for key, value in raw_meta.items():
            meta[f"meta_{key}"] = clean_metadata_value(value)

    return meta

ids, documents, metadatas = [], [], []

for item in chunks:
    # id trong file của bạn đã ổn định, dùng trực tiếp làm Chroma ID.
    doc_id = str(item["id"])

    # text là nội dung chunk dùng để embed và làm context trả lời.
    text = str(item.get("text", "")).strip()

    # Bỏ qua chunk rỗng nếu có lỗi bất thường.
    if not text:
        continue

    ids.append(doc_id)
    documents.append(text)
    metadatas.append(build_metadata(item))

print("Số ids:", len(ids))
print("Số documents:", len(documents))
print("Số metadatas:", len(metadatas))
print("Số ID unique:", len(set(ids)))
assert len(ids) == len(set(ids)), "Có ID bị trùng, cần kiểm tra lại file JSONL."

print("Metadata mẫu:")
print(json.dumps(metadatas[0], ensure_ascii=False, indent=2)[:1500])


# **7. Hàm cache embedding**



=> tạo embedding cho từng chunk và lưu cache để hạn chế gọi lại API.

In [ ]:
def load_embedding_cache(cache_path: str) -> Dict[str, List[float]]:
    # cache là dict: {chunk_id: embedding_vector}
    cache = {}
    path = Path(cache_path)

    # Nếu chưa có cache thì trả về dict rỗng.
    if not path.exists():
        return cache

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)

            # Chỉ dùng cache tạo bởi đúng embedding model hiện tại.
            if obj.get("model") == EMBEDDING_MODEL:
                cache[obj["id"]] = obj["embedding"]

    return cache


def append_embedding_cache(cache_path: str, doc_id: str, embedding: List[float]) -> None:
    # Append từng dòng để nếu notebook dừng thì các embedding trước đó vẫn được lưu.
    record = {"id": doc_id, "model": EMBEDDING_MODEL, "embedding": embedding}
    with open(cache_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

embedding_cache = load_embedding_cache(EMBEDDING_CACHE_PATH)
print(f"Số embedding đã có trong cache: {len(embedding_cache)}")


# **8. Hàm gọi OpenAI Embedding API**

=> tạo embedding cho 1 batch văn bản bằng OpenAI

In [ ]:
def embed_batch_openai(texts: List[str]) -> List[List[float]]:
    # Gọi OpenAI Embedding API cho danh sách texts.
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )

    # response.data giữ kết quả theo cùng thứ tự với input.
    return [item.embedding for item in response.data]

# Test nhanh để chắc API key và model hoạt động.
test_vector = embed_batch_openai(["Kiểm tra kết nối OpenAI Embedding API."])[0]
print("Độ dài vector embedding:", len(test_vector))
print("5 giá trị đầu:", test_vector[:5])


# **9. Tạo embedding cho toàn bộ chunk**

=> embedding toàn bộ chunks, dùng cache nếu có

In [ ]:
# Tìm các chunk chưa có cache.
pending_ids, pending_texts = [], []

for doc_id, text in zip(ids, documents):
    if doc_id not in embedding_cache:
        pending_ids.append(doc_id)
        pending_texts.append(text)

print("Tổng số chunks:", len(ids))
print("Đã có cache:", len(ids) - len(pending_ids))
print("Cần gọi API thêm:", len(pending_ids))

# Gọi API theo batch cho các chunk chưa có cache.
for start in tqdm(range(0, len(pending_texts), EMBED_BATCH_SIZE), desc="Embedding pending chunks"):
    end = start + EMBED_BATCH_SIZE
    batch_ids = pending_ids[start:end]
    batch_texts = pending_texts[start:end]

    # Tạo embedding bằng OpenAI.
    batch_embeddings = embed_batch_openai(batch_texts)

    # Lưu vào RAM cache và file cache ngay sau mỗi batch.
    for doc_id, emb in zip(batch_ids, batch_embeddings):
        embedding_cache[doc_id] = emb
        append_embedding_cache(EMBEDDING_CACHE_PATH, doc_id, emb)

    # Nghỉ nhẹ để giảm khả năng rate limit.
    time.sleep(0.2)

# Tạo danh sách embeddings đúng thứ tự với ids/documents/metadatas.
all_embeddings = [embedding_cache[doc_id] for doc_id in ids]

print("Hoàn tất embedding.")
print("Số embeddings:", len(all_embeddings))
print("Độ dài vector mẫu:", len(all_embeddings[0]))


# **10. Tạo Chroma persistent database**

=> tạo / lấy Chroma collection để lưu vector

In [ ]:
# PersistentClient lưu DB xuống thư mục CHROMA_DIR.
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# Nếu muốn build lại sạch từ đầu, bỏ comment dòng dưới.
# chroma_client.delete_collection(COLLECTION_NAME)

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={
        "project": "OU academic RAG chatbot",
        "embedding_model": EMBEDDING_MODEL,
        "data_file": JSONL_PATH
    }
)

print("Collection:", COLLECTION_NAME)
print("Số item hiện có:", collection.count())

# **11. Upsert dữ liệu vào Chroma**

=> lưu ids, documents, metadata, embeddings vào Chroma

In [ ]:
def upsert_to_chroma(batch_size: int = 64) -> None:
    # Ghi vào Chroma theo batch để ổn định hơn.
    total = len(ids)

    for start in tqdm(range(0, total, batch_size), desc="Upserting to Chroma"):
        end = start + batch_size

        # upsert giúp chạy lại không bị lỗi trùng ID.
        collection.upsert(
            ids=ids[start:end],
            documents=documents[start:end],
            metadatas=metadatas[start:end],
            embeddings=all_embeddings[start:end]
        )

upsert_to_chroma(batch_size=64)

print("Đã lưu xong Chroma DB.")
print("Tổng số item trong collection:", collection.count())


# **12. Test retrieve nhanh sau khi build**

=> kiểm tra  DB vừa build có truy xuất được không

In [ ]:
def embed_query(query: str) -> List[float]:
    # Query phải dùng cùng model embedding với documents.
    return embed_batch_openai([query])[0]


def quick_retrieve(query: str, top_k: int = 5):
    # Tạo vector câu hỏi.
    query_embedding = embed_query(query)

    # Query Chroma bằng vector câu hỏi.
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    # In kết quả để kiểm tra thủ công.
    for rank, (doc, meta, distance) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
        start=1
    ):
        print("=" * 100)
        print(f"Rank {rank} | distance = {distance:.4f}")
        print("document_type:", meta.get("document_type"))
        print("chunk_type:", meta.get("chunk_type"))
        print("source:", meta.get("document_name"))
        print("page:", meta.get("page_start"), "-", meta.get("page_end"))
        print("Preview:")
        print(doc[:700])

quick_retrieve("Sinh viên cần điều kiện gì để được xét tốt nghiệp?", top_k=5)
